# Final Competitive Markets Base Modelling Dataset

This notebook prepares the dissertation modelling dataset from:

1. `Final Polymarkets datasets(1).xlsx`
2. `Final_Yahoo_Finance_BTC_Gold_SP500_Raw.xlsx`

It creates:

- BTC vs Gold 2025 competition
- BTC vs Gold 2026 competition
- BTC vs Gold vs S&P 500 competition
- One stacked master long-format dataset
- Event IDs
- Forward log returns
- One winner per event

Advanced Polymarket features and technical indicators are NOT added yet.


## Step 1 — Upload the Two Excel Files

Upload these two files to the Colab session:

- `Final Polymarkets datasets(1).xlsx`
- `Final_Yahoo_Finance_BTC_Gold_SP500_Raw.xlsx`

You can use the Files panel on the left side of Colab, or run the upload cell below.


## Step 2 — Import Libraries and Load the Files


In [ ]:
# ============================================================
# STEP 2: Import Libraries and Load Excel Files
# ============================================================

import pandas as pd
import numpy as np

POLY_PATH = "/content/Final Polymarkets datasets.xlsx"
FIN_PATH = "/content/Final_Yahoo_Finance_BTC_Gold_SP500_Raw.xlsx"

poly = pd.read_excel(
    POLY_PATH,
    sheet_name="All Daily Wide"
)

finance = pd.read_excel(
    FIN_PATH,
    sheet_name="All Financial Data"
)

poly["Date"] = pd.to_datetime(poly["Date"])
finance["Date"] = pd.to_datetime(finance["Date"])

print("Polymarket shape:", poly.shape)
print("Financial shape:", finance.shape)

print("\nPolymarket date range:")
print(poly["Date"].min(), "to", poly["Date"].max())

print("\nFinancial date range:")
print(finance["Date"].min(), "to", finance["Date"].max())

display(poly.head())
display(finance.head())


## Step 3 — Prepare Yahoo Finance Prices

For the base dataset we keep only:

- Date
- Asset
- Adjusted Close price

The price data are then reshaped into wide format so Bitcoin, Gold and S&P 500 can be aligned by date.


In [ ]:
# ============================================================
# STEP 3: Prepare Financial Prices
# ============================================================

prices = finance[
    ["Date", "Asset", "Adj Close"]
].copy()

prices = prices.rename(
    columns={"Adj Close": "Price"}
)

duplicates = prices.duplicated(
    subset=["Date", "Asset"]
).sum()

print("Duplicate Date-Asset rows:", duplicates)

price_wide = prices.pivot(
    index="Date",
    columns="Asset",
    values="Price"
).sort_index()

print("\nAvailable price observations:")
print(price_wide.notna().sum())

display(price_wide.head(10))


## Step 4 — Create Common BTC vs Gold Trading Periods

Bitcoin trades every day, while Gold does not.

For a fair BTC vs Gold competition, only dates where BOTH assets have prices are kept.

The target return is the forward log return from the current common trading date to the next common trading date.


In [ ]:
# ============================================================
# STEP 4: Create BTC-Gold Common Trading Periods
# ============================================================

bg_prices = price_wide[
    ["Bitcoin", "Gold"]
].dropna().copy()

bg_prices["Next_Date"] = bg_prices.index.to_series().shift(-1)

bg_prices["Bitcoin_Price_t"] = bg_prices["Bitcoin"]
bg_prices["Gold_Price_t"] = bg_prices["Gold"]

bg_prices["Bitcoin_Price_Next"] = bg_prices["Bitcoin"].shift(-1)
bg_prices["Gold_Price_Next"] = bg_prices["Gold"].shift(-1)

bg_prices["Bitcoin_Return"] = np.log(
    bg_prices["Bitcoin_Price_Next"] /
    bg_prices["Bitcoin_Price_t"]
)

bg_prices["Gold_Return"] = np.log(
    bg_prices["Gold_Price_Next"] /
    bg_prices["Gold_Price_t"]
)

bg_prices = bg_prices.dropna(
    subset=[
        "Next_Date",
        "Bitcoin_Return",
        "Gold_Return"
    ]
).reset_index()

print("BTC-Gold common financial periods:", len(bg_prices))

display(
    bg_prices[
        [
            "Date",
            "Next_Date",
            "Bitcoin_Price_t",
            "Bitcoin_Price_Next",
            "Bitcoin_Return",
            "Gold_Return"
        ]
    ].head(10)
)


## Step 5 — Create Common BTC vs Gold vs S&P 500 Trading Periods

For the three-asset market, a date is kept only when:

- Bitcoin has a price
- Gold has a price
- S&P 500 has a price

The forward return is calculated to the next date when all three assets are available.


In [ ]:
# ============================================================
# STEP 5: Create BTC-Gold-SP500 Common Trading Periods
# ============================================================

three_prices = price_wide[
    ["Bitcoin", "Gold", "SP500"]
].dropna().copy()

three_prices["Next_Date"] = (
    three_prices.index.to_series().shift(-1)
)

for asset in ["Bitcoin", "Gold", "SP500"]:

    three_prices[f"{asset}_Price_t"] = (
        three_prices[asset]
    )

    three_prices[f"{asset}_Price_Next"] = (
        three_prices[asset].shift(-1)
    )

    three_prices[f"{asset}_Return"] = np.log(
        three_prices[f"{asset}_Price_Next"] /
        three_prices[f"{asset}_Price_t"]
    )

three_prices = three_prices.dropna(
    subset=[
        "Next_Date",
        "Bitcoin_Return",
        "Gold_Return",
        "SP500_Return"
    ]
).reset_index()

print(
    "BTC-Gold-SP500 common financial periods:",
    len(three_prices)
)

display(
    three_prices[
        [
            "Date",
            "Next_Date",
            "Bitcoin_Return",
            "Gold_Return",
            "SP500_Return"
        ]
    ].head(10)
)


## Step 6 — Build BTC vs Gold 2025 Events

For the binary Polymarket:

- Bitcoin probability = YES probability
- Gold probability = 1 − Bitcoin probability

Each market-date becomes one Event ID.

The winner is the asset with the highest forward log return.


In [ ]:
# ============================================================
# STEP 6A: Prepare BTC vs Gold 2025
# ============================================================

bg25 = poly[
    [
        "Date",
        "BTC_Outperform_Gold_2025_Prob"
    ]
].dropna().copy()

bg25 = bg25.rename(
    columns={
        "BTC_Outperform_Gold_2025_Prob":
        "Bitcoin_PM_Prob"
    }
)

bg25["Gold_PM_Prob"] = (
    1 - bg25["Bitcoin_PM_Prob"]
)

bg25 = bg25.merge(
    bg_prices[
        [
            "Date",
            "Next_Date",
            "Bitcoin_Price_t",
            "Bitcoin_Price_Next",
            "Bitcoin_Return",
            "Gold_Price_t",
            "Gold_Price_Next",
            "Gold_Return"
        ]
    ],
    on="Date",
    how="inner"
)

# The 2025 contract should not create a target
# whose next financial date is after 2025.
bg25 = bg25[
    bg25["Next_Date"] <= pd.Timestamp("2025-12-31")
].copy()

print(
    "BTC-Gold 2025 valid event dates:",
    len(bg25)
)

display(bg25.head())


In [ ]:
# ============================================================
# STEP 6B: Convert BTC vs Gold 2025 to Long Format
# ============================================================

rows = []

for _, r in bg25.iterrows():

    event_id = (
        "BTC_GOLD_2025_" +
        r["Date"].strftime("%Y-%m-%d")
    )

    rows.append({
        "Market_ID": "BTC_GOLD_2025",
        "Event_ID": event_id,
        "Date": r["Date"],
        "Next_Financial_Date": r["Next_Date"],
        "Asset": "Bitcoin",
        "PM_Prob": r["Bitcoin_PM_Prob"],
        "Price_t": r["Bitcoin_Price_t"],
        "Price_Next": r["Bitcoin_Price_Next"],
        "Forward_Log_Return": r["Bitcoin_Return"]
    })

    rows.append({
        "Market_ID": "BTC_GOLD_2025",
        "Event_ID": event_id,
        "Date": r["Date"],
        "Next_Financial_Date": r["Next_Date"],
        "Asset": "Gold",
        "PM_Prob": r["Gold_PM_Prob"],
        "Price_t": r["Gold_Price_t"],
        "Price_Next": r["Gold_Price_Next"],
        "Forward_Log_Return": r["Gold_Return"]
    })

bg25_long = pd.DataFrame(rows)

bg25_long["Winner"] = (
    bg25_long["Forward_Log_Return"] ==
    bg25_long.groupby("Event_ID")[
        "Forward_Log_Return"
    ].transform("max")
).astype(int)

print(
    "BTC-Gold 2025 Event IDs:",
    bg25_long["Event_ID"].nunique()
)

display(bg25_long.head(8))


## Step 7 — Build BTC vs Gold 2026 Events


In [ ]:
# ============================================================
# STEP 7: Build BTC vs Gold 2026 Competition
# ============================================================

bg26 = poly[
    [
        "Date",
        "BTC_Outperform_Gold_2026_Prob"
    ]
].dropna().copy()

bg26 = bg26.rename(
    columns={
        "BTC_Outperform_Gold_2026_Prob":
        "Bitcoin_PM_Prob"
    }
)

bg26["Gold_PM_Prob"] = (
    1 - bg26["Bitcoin_PM_Prob"]
)

bg26 = bg26.merge(
    bg_prices[
        [
            "Date",
            "Next_Date",
            "Bitcoin_Price_t",
            "Bitcoin_Price_Next",
            "Bitcoin_Return",
            "Gold_Price_t",
            "Gold_Price_Next",
            "Gold_Return"
        ]
    ],
    on="Date",
    how="inner"
)

rows = []

for _, r in bg26.iterrows():

    event_id = (
        "BTC_GOLD_2026_" +
        r["Date"].strftime("%Y-%m-%d")
    )

    rows.append({
        "Market_ID": "BTC_GOLD_2026",
        "Event_ID": event_id,
        "Date": r["Date"],
        "Next_Financial_Date": r["Next_Date"],
        "Asset": "Bitcoin",
        "PM_Prob": r["Bitcoin_PM_Prob"],
        "Price_t": r["Bitcoin_Price_t"],
        "Price_Next": r["Bitcoin_Price_Next"],
        "Forward_Log_Return": r["Bitcoin_Return"]
    })

    rows.append({
        "Market_ID": "BTC_GOLD_2026",
        "Event_ID": event_id,
        "Date": r["Date"],
        "Next_Financial_Date": r["Next_Date"],
        "Asset": "Gold",
        "PM_Prob": r["Gold_PM_Prob"],
        "Price_t": r["Gold_Price_t"],
        "Price_Next": r["Gold_Price_Next"],
        "Forward_Log_Return": r["Gold_Return"]
    })

bg26_long = pd.DataFrame(rows)

bg26_long["Winner"] = (
    bg26_long["Forward_Log_Return"] ==
    bg26_long.groupby("Event_ID")[
        "Forward_Log_Return"
    ].transform("max")
).astype(int)

print(
    "BTC-Gold 2026 Event IDs:",
    bg26_long["Event_ID"].nunique()
)

display(bg26_long.head(8))


## Step 8 — Build BTC vs Gold vs S&P 500 Events

The already-normalised three-way Polymarket probabilities are used.

Each event contains exactly three alternatives:

- Bitcoin
- Gold
- S&P 500


In [ ]:
# ============================================================
# STEP 8: Build BTC-Gold-SP500 Competition
# ============================================================

three_pm = poly[
    [
        "Date",
        "Bitcoin_Best_2026_Prob_Norm",
        "Gold_Best_2026_Prob_Norm",
        "SP500_Best_2026_Prob_Norm"
    ]
].dropna().copy()

three_pm = three_pm.merge(
    three_prices[
        [
            "Date",
            "Next_Date",
            "Bitcoin_Price_t",
            "Bitcoin_Price_Next",
            "Bitcoin_Return",
            "Gold_Price_t",
            "Gold_Price_Next",
            "Gold_Return",
            "SP500_Price_t",
            "SP500_Price_Next",
            "SP500_Return"
        ]
    ],
    on="Date",
    how="inner"
)

rows = []

for _, r in three_pm.iterrows():

    event_id = (
        "BTC_GOLD_SP500_" +
        r["Date"].strftime("%Y-%m-%d")
    )

    asset_info = {
        "Bitcoin": {
            "prob":
            r["Bitcoin_Best_2026_Prob_Norm"],
            "price_t":
            r["Bitcoin_Price_t"],
            "price_next":
            r["Bitcoin_Price_Next"],
            "return":
            r["Bitcoin_Return"]
        },
        "Gold": {
            "prob":
            r["Gold_Best_2026_Prob_Norm"],
            "price_t":
            r["Gold_Price_t"],
            "price_next":
            r["Gold_Price_Next"],
            "return":
            r["Gold_Return"]
        },
        "SP500": {
            "prob":
            r["SP500_Best_2026_Prob_Norm"],
            "price_t":
            r["SP500_Price_t"],
            "price_next":
            r["SP500_Price_Next"],
            "return":
            r["SP500_Return"]
        }
    }

    for asset, info in asset_info.items():

        rows.append({
            "Market_ID": "BTC_GOLD_SP500",
            "Event_ID": event_id,
            "Date": r["Date"],
            "Next_Financial_Date":
            r["Next_Date"],
            "Asset": asset,
            "PM_Prob": info["prob"],
            "Price_t": info["price_t"],
            "Price_Next":
            info["price_next"],
            "Forward_Log_Return":
            info["return"]
        })

three_long = pd.DataFrame(rows)

three_long["Winner"] = (
    three_long["Forward_Log_Return"] ==
    three_long.groupby("Event_ID")[
        "Forward_Log_Return"
    ].transform("max")
).astype(int)

print(
    "BTC-Gold-SP500 Event IDs:",
    three_long["Event_ID"].nunique()
)

display(three_long.head(12))


## Step 9 — Stack All Three Markets

The three prepared markets are combined into one master long-format dataset.

`Market_ID` is retained so the markets can also be analysed separately later.


In [ ]:
# ============================================================
# STEP 9: Stack All Competitive Markets
# ============================================================

master = pd.concat(
    [
        bg25_long,
        bg26_long,
        three_long
    ],
    ignore_index=True
)

master = master.sort_values(
    [
        "Date",
        "Market_ID",
        "Event_ID",
        "Asset"
    ]
).reset_index(drop=True)

print("Total rows:", len(master))
print(
    "Total Event IDs:",
    master["Event_ID"].nunique()
)

print("\nEvent IDs by market:")
print(
    master.groupby("Market_ID")[
        "Event_ID"
    ].nunique()
)

display(master.head(20))


## Step 10 — Validate the Base Dataset

Before adding features, the dataset must satisfy:

- Exactly one winner per Event ID
- Binary events contain exactly 2 assets
- Three-way events contain exactly 3 assets
- Polymarket probabilities sum to approximately 1 within each event
- No missing values in the core modelling variables


In [ ]:
# ============================================================
# STEP 10: Validate Final Base Modelling Dataset
# ============================================================

print("=" * 70)
print("FINAL BASE DATA VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Exactly one winner per Event ID
# ------------------------------------------------------------

winner_check = (
    master.groupby("Event_ID")[
        "Winner"
    ].sum()
)

print("\nWinner counts per Event ID:")
print(
    winner_check.value_counts().sort_index()
)

bad_winners = winner_check[
    winner_check != 1
]

print(
    "\nEvents without exactly one winner:",
    len(bad_winners)
)

# ------------------------------------------------------------
# 2. Choice-set sizes
# ------------------------------------------------------------

choice_size = (
    master.groupby("Event_ID")[
        "Asset"
    ].nunique()
)

print("\nChoice-set sizes:")
print(
    choice_size.value_counts().sort_index()
)

# ------------------------------------------------------------
# 3. Probability sums
# ------------------------------------------------------------

prob_sum = (
    master.groupby("Event_ID")[
        "PM_Prob"
    ].sum()
)

print("\nProbability-sum summary:")
print(prob_sum.describe())

print(
    "\nMaximum absolute deviation from 1:",
    (prob_sum - 1).abs().max()
)

# ------------------------------------------------------------
# 4. Missing values
# ------------------------------------------------------------

core_columns = [
    "Market_ID",
    "Event_ID",
    "Date",
    "Next_Financial_Date",
    "Asset",
    "PM_Prob",
    "Price_t",
    "Price_Next",
    "Forward_Log_Return",
    "Winner"
]

print("\nMissing values:")
print(
    master[core_columns].isna().sum()
)

# ------------------------------------------------------------
# 5. Summary by market
# ------------------------------------------------------------

event_summary = (
    master.groupby("Market_ID")
    .agg(
        Events=("Event_ID", "nunique"),
        Rows=("Event_ID", "size"),
        First_Date=("Date", "min"),
        Last_Date=("Date", "max")
    )
)

print("\nEvent summary:")
display(event_summary)


## Step 11 — Inspect a Few Complete Events

This is a visual check to make sure that each Event ID contains the correct competing assets and exactly one winner.


In [ ]:
# ============================================================
# STEP 11: Inspect Sample Events
# ============================================================

sample_event_ids = (
    master["Event_ID"]
    .drop_duplicates()
    .head(5)
    .tolist()
)

sample_events = master[
    master["Event_ID"].isin(
        sample_event_ids
    )
].copy()

display(
    sample_events[
        [
            "Market_ID",
            "Event_ID",
            "Date",
            "Next_Financial_Date",
            "Asset",
            "PM_Prob",
            "Forward_Log_Return",
            "Winner"
        ]
    ]
)


## Step 12 — Save the Final BASE Modelling Dataset

The Excel output contains:

- `Master_Long`
- `BTC_Gold_2025`
- `BTC_Gold_2026`
- `BTC_Gold_SP500`
- `Event_Summary`

This is the clean BASE dataset.

The next notebook will add Polymarket features and simple lagged financial features.


In [ ]:
# ============================================================
# STEP 12: Save Final Base Modelling Dataset
# ============================================================

output_path = (
    "/content/"
    "Final_Competitive_Markets_BASE_Modeling_Dataset.xlsx"
)

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    master.to_excel(
        writer,
        sheet_name="Master_Long",
        index=False
    )

    bg25_long.to_excel(
        writer,
        sheet_name="BTC_Gold_2025",
        index=False
    )

    bg26_long.to_excel(
        writer,
        sheet_name="BTC_Gold_2026",
        index=False
    )

    three_long.to_excel(
        writer,
        sheet_name="BTC_Gold_SP500",
        index=False
    )

    event_summary.to_excel(
        writer,
        sheet_name="Event_Summary"
    )

print("Saved successfully:")
print(output_path)


## Step 13 — Download the Prepared Excel Dataset


In [ ]:
# ============================================================
# STEP 13: Download Prepared Dataset
# ============================================================

from google.colab import files

files.download(
    "/content/"
    "Final_Competitive_Markets_BASE_Modeling_Dataset.xlsx"
)
